In [1]:
import copy
import dataclasses

import numpy
import scipy.spatial

import cloudvolume
import kimimaro
import fastremap

np = numpy

In [2]:
import ac_pcg.label
import ac_pcg.chunks
import ac_pcg.skeletons

from ac_pcg.pcgraph.edges import Edges
from ac_pcg.pcgraph.edges import EDGE_TYPES

from ac_pcg.io.edges import put_chunk_edges
from ac_pcg.io.components import put_chunk_components

from ac_pcg.utils import (
    label_chunk,
    chunk_edges_from_skeleton
)

In [3]:
import gzip
import pathlib
import pickle

def read_gzip_array(fn, preprocess_func=lambda x: x):
    with gzip.open(fn, "rb") as f:
        a = numpy.load(f)
    return preprocess_func(a)

test_data_path = pathlib.Path(
    "/allen/programs/celltypes/workgroups/em-connectomics/russelt/pcg_axconn/test_data_strip/"
)

test_data_labeled_array_path = test_data_path / "H17_x55_S32_230412_Pos42.npy.gz"
test_skels_path = test_data_path / "H17_x55_S32_230412_Pos42.skels.pkl"

In [4]:
labeled_array = read_gzip_array(test_data_labeled_array_path)
with test_skels_path.open(mode="rb") as skels_fobj:
    label_skels = pickle.load(skels_fobj)

In [5]:
%%time
import rtree

skel_id_to_bboxes = {sk_id: cloudvolume.Bbox.from_points(sk.vertices) for sk_id, sk in label_skels.items()}


p = rtree.index.Property()
p.dimension = 3
label_skel_idx = rtree.index.Index(
    ((sk_id, skel_bb.to_list(), label_skels[sk_id]) for sk_id, skel_bb in skel_id_to_bboxes.items()),
    properties=p
)

CPU times: user 4.28 s, sys: 83.9 ms, total: 4.36 s
Wall time: 4.36 s


In [6]:
labeled_array_bbox = cloudvolume.CloudVolume
labeled_array_chunksize = numpy.array((128, 128, 128))

In [7]:
def get_bbox_chunks(bbox, chunk_size, offset=None):
    if offset is not None:
        raise NotImplementedError
    (chunk_min, chunk_max), (remainder_min, remainder_max) = numpy.divmod(
        numpy.array([bbox.minpt, bbox.maxpt]), chunk_size)
    # chunk_max += remainder_max.astype(bool)

    # TODO could be more clever about dims
    imin, jmin, kmin = chunk_min
    imax, jmax, kmax = chunk_max
    chunks = np.mgrid[imin:imax+1:1, jmin:jmax+1:1, kmin:kmax+1:1].reshape(3, -1).T

    return chunks

In [8]:
%%time

skel_to_chunks = {sk_id: get_bbox_chunks(sk_bbox, labeled_array_chunksize) for sk_id, sk_bbox in skel_id_to_bboxes.items()}

chunk_to_skel_ids = {}
for sk_id, sk_chunks in skel_to_chunks.items():
    for sk_chunk in sk_chunks:
        try:
            chunk_to_skel_ids[tuple(sk_chunk)].append(sk_id)
        except KeyError:
            chunk_to_skel_ids[tuple(sk_chunk)] = [sk_id]

CPU times: user 2.6 s, sys: 7.76 ms, total: 2.61 s
Wall time: 2.61 s


In [9]:
def chunk_idx_to_bbox(chunk_idx, chunk_size, chunked_box_shape):
    chunk_mins = tuple(idx * chunk_d for idx, chunk_d in zip(chunk_idx, chunk_size))
    chunk_max = tuple(min(box_d, chunk_min + chunk_d) for chunk_min, chunk_d, box_d in zip(chunk_mins, chunk_size, chunked_box_shape))
    bbox = cloudvolume.Bbox(chunk_mins, chunk_max)
    return ac_pcg.chunks.ChunkBbox(
        bbox=bbox,
        chunk_idx=chunk_idx
    )


def process_oversegment_array(arr, skels, label_func, oversegment_kwargs=None, relabel_zero=False):
    oversegment_kwargs = oversegment_kwargs or {}

    oversegmented_arr, oversegmented_skels = kimimaro.utility.oversegment(
        arr, skels, **oversegment_kwargs)

    # convert labels to uint64 layer/chunk indices
    lbl_map = {
        input_lbl: label_func(input_lbl)  # input_lbl: labeler.encode_chunk_seg(chunk_box.chunk_idx, input_lbl)
        for input_lbl in fastremap.unique(
            oversegmented_arr
        )
    }
    if not relabel_zero:
        lbl_map[0] = 0

    relabeled_arr = fastremap.remap(oversegmented_arr, lbl_map)

    for skel in oversegmented_skels:
        skel.segments = fastremap.remap(skel.segments, lbl_map)

    return relabeled_arr, oversegmented_skels


def process_oversegment_bbox_shmem(bbox, sharedarray_props, *args, **kwargs):
    existing_shm = multiprocessing.shared_memory.SharedMemory(name=sharedarray_props.sharedmem_name)
    arr = numpy.ndarray(sharedarray_props.shape, dtype=sharedarray_props.dtype, buffer=existing_shm.buf)
    bbox_arr = arr[bbox.to_slices()]
    
    result = process_oversegment_array(arr, *args, **kwargs)
    existing_shm.close()
    return result

In [10]:
# run oversegmentation over all chunks

import time

chunk_size = (128, 128, 128)
chunk_boxes = ac_pcg.chunks.iterate_chunk_slice_boxes(
    labeled_array.shape, chunk_size)

labeler = ac_pcg.label.ChunkLabeler()

output_arr = numpy.empty(labeled_array.shape, dtype=numpy.uint64)

output_skels = copy.deepcopy(label_skels)

tic = time.time()
for chunk_num, chunk_box in enumerate(chunk_boxes):
    chunk_contains_bb = cloudvolume.Bbox(
        chunk_box.bbox.minpt,
        chunk_box.bbox.maxpt - 1
    )
    subvol_arr = labeled_array[chunk_box.bbox.to_slices()]
    skels_indices_tuples = filter(None, (
        ac_pcg.skeletons.bboxed_skel(res.object, chunk_contains_bb)
        for res in label_skel_idx.intersection(
            chunk_contains_bb.to_list(), objects=True
        )
    ))
    try:
        subvol_skels, subvol_skel_indices = zip(*skels_indices_tuples)
    except ValueError:
        continue

    oversegmented_subvol_arr, oversegmented_subvol_skels = kimimaro.utility.oversegment(
        subvol_arr, subvol_skels, downsample=6, progress=False)

    # convert labels to uint64 layer/chunk indices
    lbl_map = {
        input_lbl: labeler.encode_chunk_seg(chunk_box.chunk_idx, input_lbl)
        for input_lbl in fastremap.unique(
            oversegmented_subvol_arr
        )
    }
    relabeled_arr = fastremap.remap(oversegmented_subvol_arr, lbl_map)
    
    output_arr[chunk_box.bbox.to_slices()] = relabeled_arr[...]

    # map new indices to original skel vertices
    for i, (skel, subvol_indices) in enumerate(zip(oversegmented_subvol_skels, subvol_skel_indices)):
        output_skel = output_skels[skel.id]
        skel.segments = fastremap.remap(skel.segments, lbl_map)
        try:
            output_skel.segments[subvol_indices] = skel.segments
        except AttributeError:
            output_skel.add_vertex_attribute(
                "segments",
                numpy.zeros(output_skel.vertices.shape[0], dtype=numpy.uint64)
            )
            output_skel.segments[subvol_indices] = skel.segments
    

    if not chunk_num % 10:
        print(chunk_num, time.time() - tic, flush=True)

print(time.time() - tic)

90 0.32550740242004395
100 1.0618896484375
120 3.5245420932769775
130 5.09771728515625
140 6.521191358566284
150 8.077763795852661
160 9.533581256866455
170 11.037976503372192
180 12.834211111068726
190 14.765043020248413
200 16.319686889648438
210 18.00233221054077
220 19.983328104019165
230 21.742926836013794
240 23.516821146011353
250 25.40552592277527
260 27.12801480293274
270 29.498092651367188
280 32.166276693344116
290 34.614646911621094
300 37.25195384025574
310 39.85396695137024
320 41.78454804420471
330 43.654690980911255
340 45.298622846603394
350 46.87965416908264
360 49.020658016204834
370 51.260849475860596
390 54.92458748817444
400 57.269779443740845
420 61.1667058467865
430 63.036654472351074
450 66.96907615661621
460 69.0696816444397
480 72.70756793022156
490 74.45710563659668
500 76.08480978012085
510 77.72208166122437
520 79.44061660766602
530 80.95708155632019
540 82.82406640052795
550 84.59016466140747
560 86.00863933563232
570 87.4272563457489
580 88.9658143520355

In [11]:
%%time

@dataclasses.dataclass
class VtxIdxLoc:
    vertices: numpy.ndarray
    indices: numpy.ndarray
    locations: numpy.ndarray


def skeleton_unique_vtxs_idxs_locs(skel):
    vtxs, idxs = fastremap.unique(skel.segments, return_index=True)
    locs = skel.vertices[idxs]
    return VtxIdxLoc(vtxs, idxs, locs)


skel_id_to_unique_vtxs_idxs_locs = {
    skel_id: skeleton_unique_vtxs_idxs_locs(skel) for skel_id, skel in output_skels.items()
}

CPU times: user 4.75 s, sys: 24 ms, total: 4.77 s
Wall time: 4.77 s


In [12]:
%%time

skel_id_to_unique_vtxs_idxs_locs = {}
skel_id_to_full_array_idxs = {}

offset = 0

for skel_id, skel in output_skels.items():
    skel_id_to_unique_vtxs_idxs_locs[skel_id] = skeleton_unique_vtxs_idxs_locs(skel)
    vtx_size = skel_id_to_unique_vtxs_idxs_locs[skel_id].vertices.size
    skel_id_to_full_array_idxs[skel_id] = numpy.arange(offset, offset + vtx_size)
    offset += vtx_size
    

CPU times: user 5.25 s, sys: 16.1 ms, total: 5.27 s
Wall time: 5.27 s


In [13]:
%%time

all_vtxs, all_idxs, all_locs = zip(*((v.vertices, v.indices, v.locations) for k, v in skel_id_to_unique_vtxs_idxs_locs.items()))

all_vtxs = numpy.concatenate(all_vtxs)
all_idxs = numpy.concatenate(all_idxs)
all_locs = numpy.concatenate(all_locs)

all_kdtree = scipy.spatial.KDTree(all_locs)

CPU times: user 205 ms, sys: 15.7 ms, total: 221 ms
Wall time: 219 ms


In [14]:
%%time
# generate edges between skeletons enforcing non-directional uniqueness

distance = 30
edge_set = set()

for i, (skel_id, vtxs_idxs_locs) in enumerate(skel_id_to_unique_vtxs_idxs_locs.items()):
    skel_tree = scipy.spatial.KDTree(vtxs_idxs_locs.locations)
    pairs = skel_tree.query_ball_tree(all_kdtree, r=distance)
    in_skel_ids = set(skel_id_to_full_array_idxs[skel_id])

    for skel_idx, pair_result in enumerate(pairs):
        skel_vtx = vtxs_idxs_locs.vertices[skel_idx]
        edge_set |= {
            frozenset((skel_vtx, all_vtxs[query_vtx_idx]))
            for query_vtx_idx in (set(pair_result) - in_skel_ids)
            if (skel_vtx != all_vtxs[query_vtx_idx])
        }
    if not i % 1000:
        print(i)

0
1000
2000
3000
4000
5000
6000
7000
8000
9000
10000
11000
12000
13000
14000
15000
16000
17000
18000
19000
20000
21000
22000
23000
24000
25000
26000
27000
28000
29000
30000
31000
32000
33000
34000
35000
36000
37000
38000
39000
40000
41000
42000
43000
44000
CPU times: user 36.8 s, sys: 961 ms, total: 37.8 s
Wall time: 37.7 s


In [15]:
%time all_pairs = numpy.array([tuple(edge) for edge in edge_set])
all_pairs.shape

CPU times: user 8.9 s, sys: 340 ms, total: 9.24 s
Wall time: 9.22 s


(7943308, 2)

In [16]:
# %%time
# # generate edges
# 
# distance = 30
# all_pairs =  []
# 
# for i, (skel_id, vtxs_idxs_locs) in enumerate(skel_id_to_unique_vtxs_idxs_locs.items()):
#     skel_tree = scipy.spatial.KDTree(vtxs_idxs_locs.locations)
#     pairs = skel_tree.query_ball_tree(all_kdtree, r=distance)
#     for skel_idx, pair_result in enumerate(pairs):
#         vtx = vtxs_idxs_locs.vertices[skel_idx]
#         pair_result = numpy.array(pair_result)
#         pair_result = pair_result[
#            ~numpy.isin(pair_result, skel_id_to_full_array_idxs[skel_id])
#         ]
#         pair_result = numpy.c_[all_vtxs[pair_result], numpy.full(pair_result.shape, vtx)]
#         all_pairs.append(pair_result)
#     if not i % 1000:
#         print(i)

In [17]:
# %time all_pairs = numpy.concatenate(all_pairs)
# %time all_pairs = numpy.sort(all_pairs, axis=1)
# %time all_pairs = fastremap.unique(all_pairs, axis=0)

In [18]:
all_pairs.shape

(7943308, 2)

In [20]:
# import fixes...
import importlib
importlib.reload(ac_pcg.utils)

<module 'ac_pcg.utils' from '/home/russelt/Dropbox/gitstuff/Coding/axconn_pcg_test/ac_pcg/utils.py'>

In [123]:
@dataclasses.dataclass
class ChunkEdgeComponentResult:
    in_chunk_edges: numpy.ndarray
    between_chunk_edges: numpy.ndarray
    chunk_components: numpy.ndarray


@dataclasses.dataclass
class ChunkEdgeResult:
    in_chunk_edges: numpy.ndarray
    between_chunk_edges: numpy.ndarray


def filter_edges_by_chunk(edge_array, query_chunk, labeler):
    segid_bits = labeler.segid_bits
    query_layer_chunk_idx = label_chunk(labeler, query_chunk)

    layer_chunk_edges = edge_array >> segid_bits
    query_chunk_edges_mask = (layer_chunk_edges == query_layer_chunk_idx)
    in_chunk_edges_mask = query_chunk_edges_mask[:, 0] & query_chunk_edges_mask[:, 1]
    between_chunk_edges_mask = query_chunk_edges_mask[:, 0] ^ query_chunk_edges_mask[:, 1]

    in_chunk_edges = edge_array[in_chunk_edges_mask]
    between_chunk_edges = edge_array[between_chunk_edges_mask]

    return ChunkEdgeResult(
        in_chunk_edges=in_chunk_edges,
        between_chunk_edges=between_chunk_edges
    )


def chunk_edges_components_from_skeleton(skel, query_chunk, labeler):
    seg_edges = skel.segments[skel.edges]
    reduced_seg_edges = fastremap.unique(
        numpy.sort(
            seg_edges[
            (seg_edges[:, 0] != seg_edges[:, 1])
            ], axis=1
        ),
        axis=0)

    segid_bits = labeler.segid_bits

    # calculate layer and chunk label for edges
    layer_chunk_edges = reduced_seg_edges >> segid_bits

    # calculate chunk label and find occurences
    query_layer_chunk_idx = label_chunk(labeler, query_chunk)
    query_chunk_edges_mask = (layer_chunk_edges == query_layer_chunk_idx)

    in_chunk_edges_mask = query_chunk_edges_mask[:, 0] & query_chunk_edges_mask[:, 1]
    between_chunk_edges_mask = query_chunk_edges_mask[:, 0] ^ query_chunk_edges_mask[:, 1]

    in_chunk_edges = reduced_seg_edges[in_chunk_edges_mask]
    between_chunk_edges = reduced_seg_edges[between_chunk_edges_mask]

    unique_components = fastremap.unique(numpy.concatenate((in_chunk_edges, between_chunk_edges), dtype=numpy.int64))
    chunk_components = (unique_components if unique_components.size else None)
    # if unique_components.size:
    #     chunk_components = numpy.concatenate(([unique_components.size], unique_components), dtype=numpy.int64)
    # else:
    #     chunk_components = None

    return ChunkEdgeComponentResult(
        in_chunk_edges=in_chunk_edges,
        between_chunk_edges=between_chunk_edges,
        chunk_components=chunk_components)

In [126]:
%%time
edge_output_loc = "file:///allen/programs/celltypes/workgroups/em-connectomics/russelt/ac_pcg_example/H17_x55_S32_230412_Pos42/example_run/edges"
component_output_loc = "file:///allen/programs/celltypes/workgroups/em-connectomics/russelt/ac_pcg_example/H17_x55_S32_230412_Pos42/example_run/components"

for chunk, chunk_skel_ids in chunk_to_skel_ids.items():
    in_chunk_edge_results = []
    between_chunk_edge_results = []
    connected_components_results = []
    # get connected components and active edges for all skeletons
    for skel_id in chunk_skel_ids:
        skel = output_skels[skel_id]
        chunk_edge_components = chunk_edges_components_from_skeleton(skel, chunk, labeler)

        if chunk_edge_components.in_chunk_edges is not None:
            in_chunk_edge_results.append(chunk_edge_components.in_chunk_edges)
        if chunk_edge_components.between_chunk_edges is not None:
            between_chunk_edge_results.append(chunk_edge_components.between_chunk_edges)
        if chunk_edge_components.chunk_components is not None:
            connected_components_results.append(chunk_edge_components.chunk_components)

    # get inactive edges from spatial query results
    chunk_inactive_edges = filter_edges_by_chunk(all_pairs, chunk, labeler)

    in_chunk_edge_results.append(chunk_inactive_edges.in_chunk_edges)
    between_chunk_edge_results.append(chunk_inactive_edges.between_chunk_edges)
    
    in_chunk_edge_arr = numpy.concatenate(in_chunk_edge_results)
    between_chunk_edge_arr = numpy.concatenate(between_chunk_edge_results)

    in_chunk_edges = Edges(*in_chunk_edge_arr.T)
    between_chunk_edges = Edges(*between_chunk_edge_arr.T)
    # cross_chunk_edges are not generated in this segmentation method
    cross_chunk_edges = Edges([], [])

    # produce chunk edge format
    edges_d = {
        EDGE_TYPES.in_chunk: in_chunk_edges,
        EDGE_TYPES.between_chunk: between_chunk_edges,
        EDGE_TYPES.cross_chunk: cross_chunk_edges
    }

    # serialization handles writing connected components format
    connected_components = connected_components_results

    has_edges = len(in_chunk_edge_results) or len(between_chunk_edge_results)
    has_components = len(connected_components)

    if has_edges:
        put_chunk_edges(edge_output_loc, chunk, edges_d, compression_level=22)
    if has_components:
        put_chunk_components(component_output_loc, connected_components, chunk)

CPU times: user 3min, sys: 15.2 s, total: 3min 15s
Wall time: 3min 26s


In [128]:
label_output_loc = "file:///allen/programs/celltypes/workgroups/em-connectomics/russelt/ac_pcg_example/H17_x55_S32_230412_Pos42/example_run/labels"
cv_info = {
    "data_type": "uint64",
    "num_channels": 1,
    "scales": [
        {
            "chunk_sizes": [
                [
                    128,
                    128,
                    128
                ]
            ],
            "compressed_segmentation_block_size": (8, 8, 8),
            "encoding": "compressed_segmentation",
            "key": "1000_1000_1000",
            "resolution": [
                1000,
                1000,
                1000
            ],
            "size": output_arr.shape,
            "voxel_offset": [
                0,
                0,
                0
            ]
        }
    ],
    "type": "segmentation"
}

In [129]:
seg_cv = cloudvolume.CloudVolume(label_output_loc, info=cv_info)


In [130]:
seg_cv.commit_info()
seg_cv[..., 0] = output_arr[...]

Uploading: 100%|████████████████████████████████████████████████████████████████████| 1521/1521 [00:46<00:00, 32.53it/s]


In [33]:
# TODO filter edges by chunk idx

In [ ]:
# put chunk edges & components